В этом файле представлена предобработка сырого текстового файла. В результате работы кода получается корпус текстов, представляющий собой json-файл, в котором хранится список списков лемм.

# Установка и импорт необходимых библиотек

In [4]:
# !python -m pip install nltk
# !python -m pip install pymorphy3
# !python -m pip install razdel

In [1]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pymorphy3
from tqdm import tqdm
import json
from razdel import sentenize

In [2]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words('russian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gigab\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gigab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gigab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Функции для обработки текста

In [15]:
def preprocess_text(books_data_path='books_data.json'):
    '''
    Функция, которая по сырому текстовому файлу создаёт структурированный корпус
    с предобработанными текстами.
    На вход: можно подать параметр books_data_path -- путь к json файлу с текстами и метаинформацией.
    На выходе: структурированный корпус текстов (список списков с леммами + метаинформация).
    '''
    with open(books_data_path, 'r', encoding='utf-8') as f:
        books_data = json.load(f)

    corpus = []

    for book_chapter in tqdm(books_data):
        book_url = book_chapter['book_url']
        book_title = book_chapter['book_title']
        chapter_name = book_chapter['chapter_name']
        chapter_text = book_chapter['chapter_text']
        chapter_text_lemmas = []
        sentences = [sentence.text for sentence in sentenize(chapter_text)]

        for sentence in sentences:
            sentence = sentence.lower()  # приводим к нижнему регистру
            sentence = re.sub(r'[^а-яёa-z\s]', ' ', sentence)  # удаляем спец символы
            sentence = re.sub(r'\s+', ' ', sentence).strip() # удаляем лишние пробелы
            tokens = word_tokenize(sentence, language='russian') # производим токенизацию

            # удаляем стоп-слова
            filtered_tokens = [t for t in tokens if t not in stop_words]

            # получаем леммы
            chapter_text_lemmas_sent = []
            for token in filtered_tokens:
                try:
                    lemma = morph.parse(token)[0].normal_form
                    chapter_text_lemmas_sent.append(lemma)
                except:
                    continue  
            chapter_text_lemmas.append(chapter_text_lemmas_sent)
    
        corpus.append({
            'book_url': book_url,
            'book_title': book_title,
            'chapter_name': chapter_name,
            'chapter_text': chapter_text,
            'chapter_text_sent_lemmas': chapter_text_lemmas,
            'chapter_sentences': sentences
        })
     
    return corpus

Итого (какая предобработка сделана):

- токенизация с помощью word_tokenize из nltk

- лемматизация

- чистка от пунктуации

- чистка от стоп-слов

- приведение к нижнему регистру

- удаление прочих спец. символов (например, английских / французских слов)

# Обработка текстов и создание корпуса

In [16]:
# запускаем функцию предобработки
corpus = preprocess_text()

100%|██████████| 239/239 [00:13<00:00, 17.35it/s]


In [17]:
# результат (список списков лемм) сохраняем в json-формате
with open('corpus.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)